In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import time

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

class PrunableLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        self.weight      = nn.Parameter(torch.empty(out_features, in_features))
        self.bias        = nn.Parameter(torch.zeros(out_features))
        self.gate_scores = nn.Parameter(torch.zeros(out_features, in_features))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates          = torch.sigmoid(self.gate_scores)
        pruned_weights = self.weight * gates
        return nn.functional.linear(x, pruned_weights, self.bias)

    def get_gates(self) -> torch.Tensor:
        return torch.sigmoid(self.gate_scores).detach().cpu()

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}"

class SelfPruningNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers  = nn.ModuleList([
            PrunableLinear(3072, 1024),
            PrunableLinear(1024,  512),
            PrunableLinear( 512,  256),
            PrunableLinear( 256,   10),
        ])
        self.bns  = nn.ModuleList([
            nn.BatchNorm1d(1024),
            nn.BatchNorm1d(512),
            nn.BatchNorm1d(256),
        ])
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)
        for i, layer in enumerate(self.layers[:-1]):
            x = self.relu(self.bns[i](layer(x)))
        return self.layers[-1](x)

    def prunable_layers(self):
        return [m for m in self.modules() if isinstance(m, PrunableLinear)]

    def gate_params(self):
        return [layer.gate_scores for layer in self.prunable_layers()]

    def weight_params(self):
        gate_ids = {id(p) for p in self.gate_params()}
        return [p for p in self.parameters() if id(p) not in gate_ids]

    def sparsity_loss(self) -> torch.Tensor:
        gates_all = torch.cat([
            torch.sigmoid(layer.gate_scores).reshape(-1)
            for layer in self.prunable_layers()
        ])
        return gates_all.mean()

    def overall_sparsity(self, threshold: float = 0.01) -> float:
        pruned = total = 0
        for layer in self.prunable_layers():
            g = layer.get_gates()
            pruned += (g < threshold).sum().item()
            total  += g.numel()
        return pruned / total if total > 0 else 0.0

    def total_gates(self) -> int:
        return sum(l.gate_scores.numel() for l in self.prunable_layers())

def get_cifar10_loaders(batch_size: int = 256):
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    train_set = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
    test_set  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
    kw = dict(num_workers=2, pin_memory=True)
    return (
        torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True,  **kw),
        torch.utils.data.DataLoader(test_set,  batch_size=batch_size, shuffle=False, **kw),
    )

def train_one_epoch(model, loader, weight_opt, gate_opt, criterion, lam):
    model.train()
    totals = defaultdict(float)

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        weight_opt.zero_grad()
        gate_opt.zero_grad()

        logits      = model(images)
        cls_loss    = criterion(logits, labels)
        sparse_loss = model.sparsity_loss()

        loss = cls_loss + lam * sparse_loss
        loss.backward()

        weight_opt.step()
        gate_opt.step()

        totals["total"]  += loss.item()
        totals["cls"]    += cls_loss.item()
        totals["sparse"] += sparse_loss.item()

    n = len(loader)
    return totals["total"]/n, totals["cls"]/n, totals["sparse"]/n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits      = model(images)
        total_loss += criterion(logits, labels).item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total


def train_model(lam: float, epochs: int = 30, batch_size: int = 256):
    print(f"\n{'='*60}")
    print(f"  Training  lambda = {lam}   (epochs={epochs})")
    print(f"{'='*60}")

    train_loader, test_loader = get_cifar10_loaders(batch_size)
    model     = SelfPruningNet().to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    weight_opt = optim.Adam(model.weight_params(), lr=1e-3, weight_decay=1e-4)
    gate_opt   = optim.Adam(model.gate_params(),   lr=0.1)

    weight_sched = optim.lr_scheduler.CosineAnnealingLR(weight_opt, T_max=epochs)
    gate_sched   = optim.lr_scheduler.CosineAnnealingLR(gate_opt,   T_max=epochs)

    history = defaultdict(list)
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        tr_total, tr_cls, tr_sparse = train_one_epoch(
            model, train_loader, weight_opt, gate_opt, criterion, lam)
        val_loss, val_acc = evaluate(model, test_loader, criterion)
        sparsity = model.overall_sparsity()
        weight_sched.step()
        gate_sched.step()

        history["val_acc"].append(val_acc)
        history["sparsity"].append(sparsity)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs} | "
                  f"Loss {tr_total:.4f} (cls {tr_cls:.4f} | sparse {tr_sparse:.4f}) | "
                  f"Val {val_acc:.2f}% | Sparsity {sparsity*100:.1f}% | "
                  f"{time.time()-t0:.0f}s")

    _, final_acc   = evaluate(model, test_loader, criterion)
    final_sparsity = model.overall_sparsity()
    pruned         = int(final_sparsity * model.total_gates())
    print(f"\n  Done ->  Accuracy: {final_acc:.2f}%  |  "
          f"Sparsity: {final_sparsity*100:.2f}%  |  "
          f"Pruned: {pruned:,} / {model.total_gates():,} gates")

    return {
        "lam": lam, "model": model, "history": dict(history),
        "final_acc": final_acc, "final_sparsity": final_sparsity,
    }

def plot_results(results: list, save_path: str = "pruning_results.png"):
    lambdas    = [r["lam"] for r in results]
    accuracies = [r["final_acc"] for r in results]
    sparsities = [r["final_sparsity"] * 100 for r in results]

    fig = plt.figure(figsize=(16, 12))
    fig.patch.set_facecolor("#0d1117")
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
    ACCENT = ["#58a6ff", "#3fb950", "#f78166"]
    BG, TEXT = "#161b22", "#c9d1d9"

    def style_ax(ax, title):
        ax.set_facecolor(BG)
        ax.set_title(title, color=TEXT, fontsize=12, pad=10, fontweight="bold")
        ax.tick_params(colors=TEXT)
        for spine in ax.spines.values():
            spine.set_edgecolor("#30363d")

    ax1 = fig.add_subplot(gs[0, 0])
    style_ax(ax1, "Test Accuracy vs Lambda")
    bars = ax1.bar([str(l) for l in lambdas], accuracies, color=ACCENT,
                   edgecolor="#30363d", linewidth=0.8)
    ax1.set_xlabel("Lambda", color=TEXT)
    ax1.set_ylabel("Test Accuracy (%)", color=TEXT)
    ax1.set_ylim(0, max(accuracies) * 1.2)
    for bar, v in zip(bars, accuracies):
        ax1.text(bar.get_x() + bar.get_width()/2, v + 0.4, f"{v:.1f}%",
                 ha="center", color=TEXT, fontsize=10, fontweight="bold")

    ax2 = fig.add_subplot(gs[0, 1])
    style_ax(ax2, "Sparsity Level vs Lambda")
    bars2 = ax2.bar([str(l) for l in lambdas], sparsities, color=ACCENT,
                    edgecolor="#30363d", linewidth=0.8)
    ax2.set_xlabel("Lambda", color=TEXT)
    ax2.set_ylabel("Sparsity (%)", color=TEXT)
    ax2.set_ylim(0, 115)
    for bar, v in zip(bars2, sparsities):
        ax2.text(bar.get_x() + bar.get_width()/2, v + 1.5, f"{v:.1f}%",
                 ha="center", color=TEXT, fontsize=10, fontweight="bold")

    ax3 = fig.add_subplot(gs[1, 0])
    style_ax(ax3, "Validation Accuracy over Epochs")
    for r, color in zip(results, ACCENT):
        ax3.plot(range(1, len(r["history"]["val_acc"]) + 1),
                 r["history"]["val_acc"], color=color, linewidth=2,
                 label=f"lambda={r['lam']}")
    ax3.set_xlabel("Epoch", color=TEXT)
    ax3.set_ylabel("Accuracy (%)", color=TEXT)
    ax3.legend(facecolor=BG, edgecolor="#30363d", labelcolor=TEXT)

    low = min(results, key=lambda r: r["lam"])
    ax4 = fig.add_subplot(gs[1, 1])
    style_ax(ax4, f"Gate Distribution  (lambda={low['lam']})")
    all_gates = np.concatenate([
        layer.get_gates().numpy().ravel()
        for layer in low["model"].prunable_layers()
    ])
    ax4.hist(all_gates, bins=120, color="#58a6ff", edgecolor="none", alpha=0.85)
    ax4.axvline(x=0.01, color="#f78166", linestyle="--", linewidth=1.5,
                label="threshold=0.01")
    ax4.set_xlabel("Gate Value", color=TEXT)
    ax4.set_ylabel("Count", color=TEXT)
    ax4.legend(facecolor=BG, edgecolor="#30363d", labelcolor=TEXT)
    pct = (all_gates < 0.01).mean() * 100
    ax4.text(0.52, 0.85,
             f"{pct:.1f}% gates < 0.01\n(effectively pruned)",
             transform=ax4.transAxes, color=TEXT, fontsize=9,
             bbox=dict(boxstyle="round", facecolor=BG, edgecolor="#30363d"))

    fig.suptitle("Self-Pruning Neural Network — CIFAR-10 Results",
                 color=TEXT, fontsize=15, fontweight="bold", y=0.98)
    plt.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    print(f"\nPlot saved -> {save_path}")
    plt.close(fig)

if __name__ == "__main__":
    LAMBDAS = [0.5, 1.0, 2.0]
    EPOCHS  = 30

    results = []
    for lam in LAMBDAS:
        results.append(train_model(lam=lam, epochs=EPOCHS))

    print("\n\n" + "="*60)
    print("  RESULTS SUMMARY")
    print("="*60)
    print(f"  {'Lambda':<10} {'Test Accuracy':>15} {'Sparsity (%)':>15}")
    print("  " + "-"*40)
    for r in results:
        print(f"  {r['lam']:<10} {r['final_acc']:>14.2f}%"
              f" {r['final_sparsity']*100:>14.2f}%")
    print("="*60)

    plot_results(results, save_path="pruning_results.png")
    print("\nDone")

Using device: cuda

  Training  lambda = 0.5   (epochs=30)
  Epoch   1/30 | Loss 1.9542 (cls 1.7586 | sparse 0.3912) | Val 43.76% | Sparsity 0.0% | 18s
  Epoch   5/30 | Loss 1.4207 (cls 1.3524 | sparse 0.1366) | Val 54.43% | Sparsity 18.2% | 95s
  Epoch  10/30 | Loss 1.2567 (cls 1.2104 | sparse 0.0925) | Val 58.41% | Sparsity 52.2% | 192s
  Epoch  15/30 | Loss 1.1663 (cls 1.1271 | sparse 0.0784) | Val 60.36% | Sparsity 70.6% | 290s
  Epoch  20/30 | Loss 1.1067 (cls 1.0702 | sparse 0.0731) | Val 61.84% | Sparsity 77.7% | 385s
  Epoch  25/30 | Loss 1.0652 (cls 1.0294 | sparse 0.0714) | Val 63.06% | Sparsity 80.1% | 483s
  Epoch  30/30 | Loss 1.0541 (cls 1.0185 | sparse 0.0711) | Val 63.12% | Sparsity 80.5% | 580s

  Done ->  Accuracy: 63.12%  |  Sparsity: 80.53%  |  Pruned: 3,063,154 / 3,803,648 gates

  Training  lambda = 1.0   (epochs=30)
  Epoch   1/30 | Loss 2.0928 (cls 1.7564 | sparse 0.3364) | Val 43.81% | Sparsity 0.4% | 20s
  Epoch   5/30 | Loss 1.4395 (cls 1.3505 | sparse 0.0890